In [29]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [30]:
pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


In [31]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [32]:
pip install seaborn


Note: you may need to restart the kernel to use updated packages.


In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [56]:
import kagglehub

# Download selected version
path = kagglehub.dataset_download("chaitu20/ipl-dataset2008-2025/versions/1")

print("Path to dataset files:", path)

Path to dataset files: /home/hopelessnerd/.cache/kagglehub/datasets/chaitu20/ipl-dataset2008-2025/versions/1


In [35]:
import os
print(os.listdir('/home/hopelessnerd/.cache/kagglehub/datasets/chaitu20/ipl-dataset2008-2025/versions/1'))

['IPL.csv']


In [36]:
# Store the path returned by kagglehub
dataset_path = r'/home/hopelessnerd/.cache/kagglehub/datasets/chaitu20/ipl-dataset2008-2025/versions/1'

In [57]:
# Combine the folder path with the exact file name
csv_file_path = os.path.join(dataset_path,"IPL.csv" )  # Replace with the actual file name printed above

# Load the data
df = pd.read_csv(csv_file_path)

# Preview the first 5 rows
df.head()

/tmp/ipykernel_29263/1186300335.py:5: DtypeWarning: Columns (28: review_batter, 29: team_reviewed, 30: review_decision, 31: umpire, 43: season, 46: superover_winner, 47: result_type, 48: method, 51: event_match_no) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file_path)


,Unnamed: 0,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_runs,team_balls,team_wicket,new_batter,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out
0,131970,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,1,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
1,131971,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,...,1,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
2,131972,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
3,131973,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,3,0,NaN,0,2,0,"('BB McCullum', 'SC Ganguly')",NaN,False
4,131974,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,...,2,4,0,NaN,0,3,0,"('BB McCullum', 'SC Ganguly')",NaN,False


In [38]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 278205 entries, 0 to 278204
Data columns (total 64 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Unnamed: 0         278205 non-null  int64  
 1   match_id           278205 non-null  int64  
 2   date               278205 non-null  str    
 3   match_type         278205 non-null  str    
 4   event_name         278205 non-null  str    
 5   innings            278205 non-null  int64  
 6   batting_team       278205 non-null  str    
 7   bowling_team       278205 non-null  str    
 8   over               278205 non-null  int64  
 9   ball               278205 non-null  int64  
 10  ball_no            278205 non-null  float64
 11  batter             278205 non-null  str    
 12  bat_pos            278205 non-null  int64  
 13  runs_batter        278205 non-null  int64  
 14  balls_faced        278205 non-null  int64  
 15  bowler             278205 non-null  str    
 16  valid_ball   

In [39]:

# Check dataset shape
print("Dataset Shape:", df.shape)

Dataset Shape: (278205, 64)


In [40]:

# Extract unique matches
matches_df = df.drop_duplicates(subset=['match_id']).copy()

print(f"Total Unique Matches: {len(matches_df)}")

Total Unique Matches: 1169


In [41]:
# Mapping dictionary for team name updates
team_mapping = {
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Deccan Chargers': 'Sunrisers Hyderabad'
}

# Replace team names across all relevant columns
team_cols = ['batting_team', 'bowling_team', 'toss_winner', 'match_won_by']
for col in team_cols:
    if col in matches_df.columns:
        matches_df[col] = matches_df[col].replace(team_mapping)


In [42]:

# 1. Fill missing winners (e.g., abandoned or rained out matches)
matches_df['match_won_by'] = matches_df['match_won_by'].fillna('No Result')

# 2. Fill missing outcome details
if 'win_outcome' in matches_df.columns:
    matches_df['win_outcome'] = matches_df['win_outcome'].fillna('No Result')

# 3. Fill missing player of the match
if 'player_of_match' in matches_df.columns:
    matches_df['player_of_match'] = matches_df['player_of_match'].fillna('None')

# 4. Fill missing city based on venue
matches_df['city'] = matches_df['city'].fillna('Unknown')

In [43]:
# Convert date to datetime format
matches_df['date'] = pd.to_datetime(matches_df['date'])

# Verify season is formatted cleanly
matches_df['season'] = matches_df['season'].astype(str)


In [44]:
# Feature: Did the toss winner win the match?
matches_df['toss_match_winner'] = (matches_df['toss_winner'] == matches_df['match_won_by']).astype(int)
# Inspect cleaned match dataframe summary
print("Cleaned Match Table Shape:", matches_df.shape)
matches_df[['match_id', 'date', 'season', 'toss_winner', 'toss_decision', 'match_won_by', 'toss_match_winner']].head()


Cleaned Match Table Shape: (1169, 65)


,match_id,date,season,toss_winner,toss_decision,match_won_by,toss_match_winner
0,335982,2008-04-18,2007/08,Royal Challengers Bangalore,field,Kolkata Knight Riders,0
225,335983,2008-04-19,2007/08,Chennai Super Kings,bat,Chennai Super Kings,1
473,335984,2008-04-19,2007/08,Rajasthan Royals,bat,Delhi Capitals,0
692,335985,2008-04-20,2007/08,Mumbai Indians,bat,Royal Challengers Bangalore,0
938,335986,2008-04-20,2007/08,Sunrisers Hyderabad,bat,Kolkata Knight Riders,0


In [45]:
import os

# Create the data folder if it doesn't exist yet
os.makedirs('data', exist_ok=True)

# Save the cleaned dataframe
matches_df.to_csv('data/cleaned_matches.csv', index=False)


In [46]:
ls data/

cleaned_matches.csv


In [47]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")

# Load pre-cleaned dataset from yesterday
matches_df = pd.read_csv('data/cleaned_matches.csv')
matches_df.head()


,Unnamed: 0,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_balls,team_wicket,new_batter,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out,toss_match_winner
0,131970,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False,0
1,132195,335983,2008-04-19,T20,Indian Premier League,1,Chennai Super Kings,Punjab Kings,0,1,...,1,0,NaN,0,1,0,"('ML Hayden', 'PA Patel')",NaN,False,1
2,132443,335984,2008-04-19,T20,Indian Premier League,1,Rajasthan Royals,Delhi Capitals,0,1,...,1,0,NaN,0,1,0,"('T Kohli', 'YK Pathan')",NaN,False,0
3,132662,335985,2008-04-20,T20,Indian Premier League,1,Mumbai Indians,Royal Challengers Bangalore,0,1,...,1,0,NaN,0,1,0,"('L Ronchi', 'ST Jayasuriya')",NaN,False,0
4,132908,335986,2008-04-20,T20,Indian Premier League,1,Sunrisers Hyderabad,Kolkata Knight Riders,0,1,...,1,0,NaN,1,1,0,"('AC Gilchrist', 'Y Venugopal Rao')",NaN,False,0


In [48]:
# PART 1: Match Phase Dynamics Analysis

In [49]:
# 1. Load the ball-by-ball dataset (e.g., Kaggle IPL deliveries dataset)
# Ensure your columns match standard naming (e.g., 'match_id', 'inning', 'over', 'ball', 
# 'batter', 'bowler', 'batsman_runs', 'extra_runs', 'total_runs', 'is_wicket', 'player_dismissed')



print(f"Total deliveries loaded: {len(df)}")

Total deliveries loaded: 278205


In [50]:
# Feature Engineering Example
def assign_phase(over):
    if over < 6:
        return 'Powerplay'
    elif over < 15:
        return 'Middle'
    else:
        return 'Death'

df['phase'] = df['over'].apply(assign_phase)
df['is_dot'] = (df['runs_total'] == 0).astype(int)
df['is_boundary'] = df['runs_batter'].isin([4, 6]).astype(int)



In [51]:
phase_analysis = df.groupby('phase').agg(
    total_runs=('runs_total', 'sum'),
    balls_faced=('ball', 'count'),
    total_wickets=('wicket_kind', 'sum'),
    dot_balls=('runs_total', lambda x: (x == 0).sum())
).reset_index()


In [52]:
# Calculate Run Rate (Runs per Over = Runs / (Balls / 6))
phase_analysis['run_rate'] = phase_analysis['total_runs'] / (phase_analysis['balls_faced'] / 6)
phase_analysis['dot_ball_pct'] = (phase_analysis['dot_balls'] / phase_analysis['balls_faced']) * 100

print("\n--- Match Phase Dynamics ---")
print(phase_analysis[['phase', 'run_rate', 'total_wickets', 'dot_ball_pct']])


--- Match Phase Dynamics ---
       phase  run_rate                                      total_wickets  \
0      Death  9.520573  caughtcaughtbowledcaughtbowledstumpedcaughtbow...   
1     Middle  7.632374  caughtcaughtcaughtcaughtrun outbowledbowledlbw...   
2  Powerplay  7.664801  caughtbowledbowledcaughtcaughtcaughtcaughtcaug...   

   dot_ball_pct  
0     27.000379  
1     31.024399  
2     44.774462  


In [ ]:
# Assumes columns: 'batter', 'bowler_type' (Pace/Spin), 'batsman_runs', 'is_wicket'


# Group by batter and bowler type
pace_spin_stats = df.groupby(['batter', 'bowler_type']).agg(
    runs=('batsman_runs', 'sum'),
    balls=('ball', 'count'),
    dismissals=('is_wicket', 'sum')
).reset_index()

# Calculate Average and Strike Rate
pace_spin_stats['batting_average'] = pace_spin_stats['runs'] / pace_spin_stats['dismissals'].replace(0, 1) # Avoid div by zero
pace_spin_stats['strike_rate'] = (pace_spin_stats['runs'] / pace_spin_stats['balls']) * 100

print(pace_spin_stats[pace_spin_stats['balls'] >= 50].head())

In [ ]:
df.info()


In [ ]:
print(df["gender"])

In [ ]:
gender_str = "male"
female = df[df["gender"] != gender_str]

# Get the count of rows
female_count = female.shape[0]
print(f"Number of female entries: {female_count}")


In [ ]:
# Assuming individual extra columns exist in the dataset
extras_df = df.groupby('season').agg(
    wides=('wide_runs', 'sum'),
    no_balls=('noball_runs', 'sum'),
    leg_byes=('legbye_runs', 'sum'),
    byes=('bye_runs', 'sum')
).reset_index()

# Melt for visualization
extras_melted = extras_df.melt(id_vars='season', value_vars=['wides', 'no_balls', 'leg_byes', 'byes'],
                               var_name='extra_type', value_name='runs')

plt.figure(figsize=(12, 6))
sns.barplot(data=extras_melted, x='season', y='runs', hue='extra_type', palette='muted')
plt.title('Extras Breakdown Across Seasons', fontsize=14, fontweight='bold')
plt.xlabel('Season', fontsize=12)
plt.ylabel('Total Runs Conceded', fontsize=12)
plt.legend(title='Extra Type')
plt.show()

In [ ]:
# Group by pre-existing 'phase' column
phase_analysis = df.groupby('phase').agg(
    total_runs=('runs_total', 'sum'),
    total_balls=('ball', 'count'), # or use 'valid_ball' if you want only legal deliveries
    total_wickets=('wicket_kind', lambda x: x.notnull().sum()),
    total_dots=('is_dot', 'sum')
).reset_index()

# Calculate Run Rate (Runs per Over = Runs / (Balls / 6))
phase_analysis['run_rate'] = phase_analysis['total_runs'] / (phase_analysis['total_balls'] / 6)
phase_analysis['dot_ball_pct'] = (phase_analysis['total_dots'] / phase_analysis['total_balls']) * 100

print(phase_analysis[['phase', 'run_rate', 'total_wickets', 'dot_ball_pct']])

In [ ]:
# Batter vs. Bowler Head-to-Head (H2H)

In [ ]:
# Group by batter and bowler
h2h = df.groupby(['batter', 'bowler']).agg(
    balls_faced=('ball', 'count'),
    runs_scored=('runs_batter', 'sum'),
    dismissals=('wicket_kind', lambda x: (x == x.notnull()).sum())
).reset_index()

# Filter for meaningful sample size (e.g., >= 30 balls faced)
significant_h2h = h2h[h2h['balls_faced'] >= 30].copy()
significant_h2h['strike_rate'] = (significant_h2h['runs_scored'] / significant_h2h['balls_faced']) * 100

top_matchups = significant_h2h.sort_values(by='runs_scored', ascending=False)
print(top_matchups[['batter', 'bowler', 'balls_faced', 'runs_scored', 'dismissals', 'strike_rate']].head(10))

In [ ]:
# Extras Breakdown

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filter rows where extras occurred (runs_extras > 0)
extras_df = df[df['runs_extras'] > 0]

# Group by season and extra_type to sum extra runs
extras_summary = extras_df.groupby(['season', 'extra_type'])['runs_extras'].sum().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=extras_summary, x='season', y='runs_extras', hue='extra_type', palette='muted')
plt.title('Extras Breakdown Across Seasons', fontsize=14, fontweight='bold')
plt.xlabel('Season', fontsize=12)
plt.ylabel('Total Extra Runs', fontsize=12)
plt.legend(title='Extra Type')
plt.show()

In [ ]:
# Count valid wickets per over
df['is_wicket'] = df['wicket_kind'].notnull().astype(int)
over_wickets = df.groupby('over')['is_wicket'].sum().reset_index()

plt.figure(figsize=(12, 5))
sns.lineplot(data=over_wickets, x='over', y='is_wicket', marker='o', color='crimson', linewidth=2.5)
plt.title('Wicket Progression Across Overs (1-20)', fontsize=14, fontweight='bold')
plt.xlabel('Over Number', fontsize=12)
plt.ylabel('Total Wickets Fallen', fontsize=12)
plt.xticks(range(1, 21))
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Calculate average boundary percentage per over
boundary_over = df.groupby('over')['is_boundary'].mean().reset_index()
boundary_over['is_boundary_pct'] = boundary_over['is_boundary'] * 100

plt.figure(figsize=(12, 5))
sns.lineplot(data=boundary_over, x='over', y='is_boundary_pct', marker='o', color='purple', linewidth=2.5)
plt.title('Boundary Frequency (4s & 6s) Across Overs (1-20)', fontsize=14, fontweight='bold')
plt.xlabel('Over Number', fontsize=12)
plt.ylabel('Boundary Percentage (%)', fontsize=12)
plt.xticks(range(1, 21))
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of Final Innings Totals (1st vs. 2nd Innings)



In [ ]:
# Get the final score for each innings in every match
final_scores = df.groupby(['match_id', 'innings'])['team_runs'].max().reset_index()

plt.figure(figsize=(10, 6))
sns.histplot(data=final_scores, x='team_runs', hue='innings', kde=True, bins=30, palette='Set2')
plt.title('Distribution of Innings Totals (1st vs. 2nd Innings)', fontsize=14, fontweight='bold')
plt.xlabel('Total Runs Scored', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(title='Inning', labels=['2nd Innings (Chase)', '1st Innings'])
plt.tight_layout()
plt.show()

In [ ]:
# Batter Profile: Total Runs vs. Strike Rate Scatter Plot

In [ ]:
# Aggregate performance per batter (filtering for players with at least 100 balls faced)
batter_summary = df.groupby('batter').agg(
    total_runs=('runs_batter', 'sum'),
    balls_faced=('ball', 'count'),
    dismissals=('wicket_kind', lambda x: (x == x.notnull()).sum())
).reset_index()

batter_summary = batter_summary[batter_summary['balls_faced'] >= 100].copy()
batter_summary['strike_rate'] = (batter_summary['total_runs'] / batter_summary['balls_faced']) * 100

plt.figure(figsize=(10, 6))
sns.scatterplot(data=batter_summary, x='strike_rate', y='total_runs', alpha=0.7, color='teal', s=60)
plt.title('Batter Profile: Total Runs vs. Strike Rate (Min 100 Balls Faced)', fontsize=14, fontweight='bold')
plt.xlabel('Strike Rate (%)', fontsize=12)
plt.ylabel('Total Runs Scored', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()